# Lesson 09 — Is My AI Too Smart?
**NSW Software Engineering Stage 6 | Software Automation**

---
### Sometimes an AI can cheat without you knowing
An AI that scores perfectly in training might be **terrible** in the real world.

It has memorised the training data instead of learning the actual pattern. This is called **overfitting**.

**Your Goals for Today:**
1. Understand the difference between **underfitting** and **overfitting**.
2. Use **training vs test error** to spot overfitting.
3. Use **cross-validation** for a more reliable accuracy estimate.

Take your time. Read the text, and click **Run** on each code box as you go down the page.

In [ ]:
# 👇 RUN THIS CELL FIRST 👇
!pip install numpy scikit-learn matplotlib --prefer-binary --quiet

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline

print("✅ Setup Complete! Move to the next section.")

---
## Concept 1: Underfitting vs Overfitting

| Problem | Also called | What's happening | Fix |
|---|---|---|---|
| **Underfitting** | High Bias | Model is too simple | Use a more complex model |
| **Good Fit** | — | Just right! | — |
| **Overfitting** | High Variance | Model memorised the training data | Use more data, simpler model |

### The Exam Analogy
- **Underfitting:** A student who barely studied — fails the practice test AND the real exam.
- **Good fit:** A student who understood the concepts — does well on both.
- **Overfitting:** A student who memorised ONLY the practice test answers — aces the practice test but fails the real exam when new questions appear.

In [ ]:
# 👇 RUN THIS CELL 👇
# Visualise the three cases

np.random.seed(1)
x = np.sort(np.random.uniform(0, 1, 25))
y = np.sin(2 * np.pi * x) + np.random.normal(0, 0.2, 25)
x_plot = np.linspace(0, 1, 200).reshape(-1,1)
X2d    = x.reshape(-1,1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
degrees = [1, 4, 15]
names   = ["Underfit (degree 1)
Too simple", "Good Fit (degree 4)
Just right ✅", "Overfit (degree 15)
Too complex"]

for ax, deg, name in zip(axes, degrees, names):
    poly  = PolynomialFeatures(degree=deg)
    Xp    = poly.fit_transform(X2d)
    m     = LinearRegression().fit(Xp, y)
    rmse  = np.sqrt(mean_squared_error(y, m.predict(Xp)))
    ax.scatter(x, y, color='steelblue', s=25, alpha=0.7)
    ax.plot(x_plot, m.predict(poly.transform(x_plot)), color='red', linewidth=2)
    ax.set_title(f"{name}
Train RMSE={rmse:.3f}", fontsize=9)
    ax.set_ylim(-2, 2)

plt.suptitle("Underfitting  |  Good Fit  |  Overfitting", fontsize=12)
plt.tight_layout(); plt.show()

---
## Concept 2: Detecting Overfitting — Train vs Test Error

**The key signal:** If training error is much lower than test error, the model is overfitting.

| Situation | Train Error | Test Error |
|---|---|---|
| Underfitting | High | High |
| Good fit | Low | Low (similar to train) |
| **Overfitting** | Very low | **Much higher than train** |

In [ ]:
# 👇 RUN THIS CELL 👇

np.random.seed(1)
x = np.sort(np.random.uniform(0, 1, 50))
y = np.sin(2*np.pi*x) + np.random.normal(0, 0.2, 50)
X2d = x.reshape(-1,1)
Xtr, Xte, ytr, yte = train_test_split(X2d, y, test_size=0.3, random_state=42)

train_rmse_list, test_rmse_list = [], []
degrees = range(1, 13)

for deg in degrees:
    pipe = make_pipeline(PolynomialFeatures(degree=deg), LinearRegression())
    pipe.fit(Xtr, ytr)
    train_rmse_list.append(np.sqrt(mean_squared_error(ytr, pipe.predict(Xtr))))
    test_rmse_list.append(np.sqrt(mean_squared_error(yte, pipe.predict(Xte))))

plt.figure(figsize=(8, 4))
plt.plot(degrees, train_rmse_list, 'o-', color='steelblue', label='Train RMSE')
plt.plot(degrees, test_rmse_list,  's-', color='tomato',    label='Test RMSE')
plt.axvline(x=4, color='green', linestyle='--', alpha=0.7, label='Sweet spot (degree 4)')
plt.xlabel('Model Complexity (Polynomial Degree)')
plt.ylabel('RMSE')
plt.title('As complexity increases, train RMSE drops but test RMSE rises')
plt.legend(); plt.tight_layout(); plt.show()
print("Notice: after degree 4, training RMSE keeps dropping BUT test RMSE goes UP — that's overfitting!")

---
## Concept 3: Cross-Validation — A More Reliable Test

A single train/test split can be "lucky" or "unlucky" depending on how the data was randomly divided.

**5-Fold Cross-Validation** divides the data into 5 groups, trains 5 times, and averages the scores. This gives a much more reliable accuracy estimate.

In [ ]:
# 👇 RUN THIS CELL 👇

np.random.seed(1)
x_cv = np.sort(np.random.uniform(0, 1, 80))
y_cv = np.sin(2*np.pi*x_cv) + np.random.normal(0, 0.2, 80)
X_cv = x_cv.reshape(-1,1)

print(f"{'Degree':>8} {'CV RMSE (mean)':>16} {'Std':>8}")
best_deg, best_score = 1, float('inf')

for deg in range(1, 10):
    pipe   = make_pipeline(PolynomialFeatures(degree=deg), LinearRegression())
    scores = cross_val_score(pipe, X_cv, y_cv, cv=5,
                             scoring='neg_root_mean_squared_error')
    mean_rmse = -scores.mean()
    std_rmse  = scores.std()
    marker = " ← best so far" if mean_rmse < best_score else ""
    print(f"{deg:>8} {mean_rmse:>16.4f} {std_rmse:>8.4f}{marker}")
    if mean_rmse < best_score:
        best_score, best_deg = mean_rmse, deg

print(f"\n✅ Best complexity by cross-validation: degree = {best_deg}")

---
## ✏️ Exercise 1: Spot the Overfitting

Look at the train and test RMSE values below.
Label each model as `"underfit"`, `"good"`, or `"overfit"`.
Replace each `None` with the correct string.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇

model_results = {
    # Example:
    "Train RMSE=0.20, Test RMSE=0.22 — very similar, both low": "good",

    # TODO: Fill in the 4 blanks:
    "Train RMSE=0.02, Test RMSE=0.85 — huge gap!": None,
    "Train RMSE=0.90, Test RMSE=0.88 — both very high": None,
    "Train RMSE=0.18, Test RMSE=0.21 — small gap, both low": None,
    "Train RMSE=0.01, Test RMSE=1.20 — test is 120x worse!": None,
}

# ⬇️ DON'T EDIT BELOW THIS LINE — THIS IS THE AUTOGRADER ⬇️
correct = {
    "Train RMSE=0.20, Test RMSE=0.22 — very similar, both low": "good",
    "Train RMSE=0.02, Test RMSE=0.85 — huge gap!": "overfit",
    "Train RMSE=0.90, Test RMSE=0.88 — both very high": "underfit",
    "Train RMSE=0.18, Test RMSE=0.21 — small gap, both low": "good",
    "Train RMSE=0.01, Test RMSE=1.20 — test is 120x worse!": "overfit",
}
score = 0
print("=== Exercise 1 Results ===")
for q, your_ans in model_results.items():
    if q == "Train RMSE=0.20, Test RMSE=0.22 — very similar, both low": continue
    ans = correct[q]
    if your_ans == ans:
        print(f"  ✅ '{ans}' — {q[:58]}")
        score += 1
    elif your_ans is None:
        print(f"  ⬜ Not answered")
    else:
        print(f"  ❌ You said '{your_ans}', correct is '{ans}'")
print(f"\nFinal Score: {score}/4")

---
## ✏️ Exercise 2: True or False about Overfitting

Replace `None` with `"True"` or `"False"`.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇

overfit_quiz = {
    # Example:
    "Overfitting means the model memorised the training data": "True",

    # TODO: Fill in the 4 blanks:
    "An overfitting model always performs well on new unseen data": None,
    "Cross-validation gives a more reliable accuracy estimate than one split": None,
    "A very low training RMSE always means a good model": None,
    "Using simpler models is one way to reduce overfitting": None,
}

# ⬇️ DON'T EDIT BELOW THIS LINE — THIS IS THE AUTOGRADER ⬇️
correct_ov = {
    "Overfitting means the model memorised the training data": "True",
    "An overfitting model always performs well on new unseen data": "False",
    "Cross-validation gives a more reliable accuracy estimate than one split": "True",
    "A very low training RMSE always means a good model": "False",
    "Using simpler models is one way to reduce overfitting": "True",
}
score2 = 0
print("=== Exercise 2 Results ===")
for q, your_ans in overfit_quiz.items():
    if q == "Overfitting means the model memorised the training data": continue
    ans = correct_ov[q]
    if your_ans == ans:
        print(f"  ✅ {ans} — {q[:62]}")
        score2 += 1
    elif your_ans is None:
        print(f"  ⬜ Not answered")
    else:
        print(f"  ❌ You said '{your_ans}', correct is '{ans}'")
print(f"\nFinal Score: {score2}/4")

---
## 🚀 EXTENSION CHALLENGES 🚀

### Exercise 3: Plot Train vs Test RMSE for the Insects Dataset

Using `latitude` to predict `wingsize` with polynomial features (degrees 1–8), plot the train vs test RMSE curves. Fill in the blanks.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇
import pandas as pd
df = pd.read_csv('insects.csv', sep='\t')
X_ins = df[['latitude']].values
y_ins = df['wingsize'].values

Xtr, Xte, ytr, yte = train_test_split(X_ins, y_ins, test_size=0.2, random_state=42)

tr_rmse_ins, te_rmse_ins = [], []
degrees_ins = range(1, 9)

for deg in degrees_ins:
    # TODO: Create a pipeline with PolynomialFeatures(degree=deg) and LinearRegression
    pipe_ins = make_pipeline(____, ____)
    pipe_ins.fit(Xtr, ytr)
    tr_rmse_ins.append(np.sqrt(mean_squared_error(ytr, pipe_ins.predict(Xtr))))
    te_rmse_ins.append(np.sqrt(mean_squared_error(yte, pipe_ins.predict(Xte))))

plt.figure(figsize=(8, 4))
# TODO: Plot both lines (train in blue, test in red)
plt.plot(degrees_ins, tr_rmse_ins, 'o-', color='steelblue', label='Train RMSE')
plt.plot(degrees_ins, te_rmse_ins, 's-', color=____, label='Test RMSE')  # fill colour
plt.xlabel('Polynomial Degree'); plt.ylabel('RMSE (mm)')
plt.title('Insects: Train vs Test RMSE')
plt.legend(); plt.tight_layout(); plt.show()

best = degrees_ins[np.argmin(te_rmse_ins)]
print(f"✅ Best degree by test RMSE: {best}")

### Exercise 4: The Debugging Challenge

The cross-validation code below has **TWO bugs**. Fix them.

In [ ]:
# 👇 FIX THE TWO BUGS IN THIS CODE 👇

np.random.seed(0)
X_bug = np.random.rand(60, 1)
y_bug = 3 * X_bug.ravel() + np.random.normal(0, 0.3, 60)

pipe_bug = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())

# BUG 1: cv=1 is not valid — cross-validation needs at least 2 folds (use 5)
scores_bug = cross_val_score(pipe_bug, X_bug, y_bug, cv=1,
                             scoring='neg_root_mean_squared_error')

# BUG 2: scores are NEGATIVE (that's how sklearn returns them) — need to negate
mean_rmse_bug = scores_bug.mean()   # ← should be -scores_bug.mean()

print(f"Cross-validated RMSE: {mean_rmse_bug:.4f}")
print("(Should be a small positive number, around 0.3)")

### Exercise 5: Fix an Overfitting Model 🏆

The code below trains a degree-12 polynomial model that is wildly overfitting.

Your task: find the **best polynomial degree** (1–10) using 5-fold cross-validation, then retrain with that degree and compare the test RMSE before and after your fix.

In [ ]:
# 👇 WRITE ALL THE CODE YOURSELF IN THIS CELL 👇
np.random.seed(5)
x5  = np.sort(np.random.uniform(0, 1, 60))
y5  = 2*x5 + 1 + np.random.normal(0, 0.2, 60)
X5  = x5.reshape(-1,1)
Xtr5, Xte5, ytr5, yte5 = train_test_split(X5, y5, test_size=0.2, random_state=0)

# This is our BAD model (degree 12 = overfitting):
bad_pipe = make_pipeline(PolynomialFeatures(degree=12), LinearRegression())
bad_pipe.fit(Xtr5, ytr5)
bad_rmse = np.sqrt(mean_squared_error(yte5, bad_pipe.predict(Xte5)))
print(f"Bad model (degree 12) test RMSE: {bad_rmse:.4f}")

# YOUR CODE: Find the best degree using cross-validation, retrain, compare




---
## ✅ Lesson 9 Complete!
You can now spot overfitting and use cross-validation to fairly evaluate your models.

**Next up:** Lesson 10 — Decision Trees & KNN. We'll learn two completely different types of AI algorithms that don't use regression at all.

---









































## 🔑 Answer Key

### Exercise 1
```
"Train 0.02, Test 0.85"   → "overfit"
"Train 0.90, Test 0.88"   → "underfit"
"Train 0.18, Test 0.21"   → "good"
"Train 0.01, Test 1.20"   → "overfit"
```

### Exercise 2
```
"overfitting performs well on new data"  → "False"
"cross-val is more reliable"             → "True"
"very low train RMSE = good model"       → "False"
"simpler models reduce overfitting"      → "True"
```

### Exercise 3
```python
pipe_ins = make_pipeline(PolynomialFeatures(degree=deg), LinearRegression())
plt.plot(degrees_ins, te_rmse_ins, 's-', color='tomato', label='Test RMSE')
```

### Exercise 4 — Two bugs fixed:
```python
scores_bug = cross_val_score(..., cv=5, ...)   # cv must be ≥ 2
mean_rmse_bug = -scores_bug.mean()              # negate because sklearn returns negative
```

### Exercise 5 — Sample answer:
```python
best_d, best_s = 1, float('inf')
for d in range(1,11):
    s = -cross_val_score(make_pipeline(PolynomialFeatures(d), LinearRegression()),
                         Xtr5, ytr5, cv=5, scoring='neg_root_mean_squared_error').mean()
    if s < best_s: best_d, best_s = d, s
good = make_pipeline(PolynomialFeatures(best_d), LinearRegression()).fit(Xtr5,ytr5)
print(f"Best degree: {best_d}, test RMSE: {np.sqrt(mean_squared_error(yte5,good.predict(Xte5))):.4f}")
```
